# Session 3 — Build a tiny chain, then break it

This is the **practice notebook**. Fill every `YOUR TURN` (`???`) while we work on the class notebook. `???` will not run until you replace it. That is on purpose. There is no answer key here.

If a word is fuzzy, open `site/handbook.html` for this session (about 10 minutes of pictures). Then come back.

A blockchain is a notebook where **every page mentions the fingerprint of
the previous page**, and someone had to do a small puzzle to publish a page.

By the end: mine a block, fake a payment, watch the verifier yell, and say
why your class coin does **not** need its own Bitcoin.



## Words we use today

Read this once. If a later cell uses the word, come back here.

| Word | In this class it means |
|---|---|
| **Block** | A page in the notebook: “here are today’s payments.” |
| **Previous hash** | This page’s sticker names the *last* page’s fingerprint. |
| **Nonce** | How many guesses until the page’s hash looks rare enough. |
| **Difficulty** | How rare “enough” is (how many leading zeros in our toy). |
| **Proof of work** | The gym beep-test: you prove you did guesses, not that you are nice. |
| **Verify** | Recheck the zeros *and* that each page points at the real last page. |
| **Unbundle** | Ask which pieces of Bitcoin you actually needed. EAGE needs almost none of them. |



## Pictures

**Example 1 — Google Doc history.** Each save has a parent. Secret-edit
last Tuesday and the history looks weird.

**Example 2 — numbered lockers.** Locker 8's sticker says "I come after
locker 7's code." Swap a jacket in locker 3 → lockers 4–8 no longer match.

**Example 3 — gym beep test.** **Difficulty** = how fast the beeps are.
A **nonce** = how many guesses until the hash *looks rare enough*
(starts with zeros, in our toy).

**Treasurer truth:** one confirmation on a public chain is **not** Fedwire.
We call that **probabilistic finality** — pretty sure, not "the Fed said so."
Also check: wrong network, admin pause, fake token contract.

**Slow walk — why one quiet edit fails.**
Page 2 says “Ava→Ben:2” and its hash starts with `000…`.
You change it to “Ava→Ben:200” and *do not re-mine*.
Two alarms:
1. The new hash no longer starts with zeros (puzzle not redone).
2. Page 3 still points at the *old* fingerprint of page 2.

That is the whole trick. Tamper-evidence is *linkage*, not magic.



### Keep this straight

**If you remember one thing.** A chain is pages that name the last page’s fingerprint. Change a middle page and the later stickers lie.

**Common mix-up.** “We should launch our own Bitcoin so Eagecoin is secure.” You need a shared testnet and a cap, not a new planet.

**Why Eagecoin cares.** EAGE is an ERC-20 *on* someone else’s chain. You inherit their referees. You still owe a mint/pause/lost-key plan.



In [ ]:
import hashlib
from dataclasses import dataclass

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

@dataclass
class Block:
    prev_hash: str
    txs: str
    nonce: int = 0

    def header(self) -> str:
        return f"{self.prev_hash}|{self.txs}|{self.nonce}"

    def hash(self) -> str:
        return sha256(self.header())

# YOUR TURN 1 -- mine: keep raising nonce until the hash starts with N zeros.

def mine(prev_hash: str, txs: str, difficulty: int = 3) -> Block:
    prefix = "0" * difficulty
    nonce = 0
    while True:
        cand = Block(prev_hash, txs, nonce)
        if ???:   # hint: cand.hash().startswith(prefix)
            print("mined nonce=", nonce, "hash=", cand.hash())
            return cand
        nonce = ???  # hint: nonce + 1

difficulty = 3
genesis = mine("0" * 64, "class opens the notebook", difficulty)
block2 = mine(genesis.hash(), "Ava->Ben:2", difficulty)
block3 = mine(block2.hash(), "Ben->Cara:1", difficulty)
chain = [genesis, block2, block3]



**Picture.** Each page names the last page's sticker. A quiet edit on page 2 leaves page 3 pointing at a ghost.



In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.4, 3.9),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Picture: locker stickers. Change page 2 and page 3 still points at the old code.
labels = ["genesis", "block 2\nAva->Ben:2", "block 3\nBen->Cara:1"]
fig, ax = plt.subplots(figsize=(7.8, 2.6))
ax.set_xlim(0, 9)
ax.set_ylim(0, 2)
ax.axis("off")
ax.set_title("Each page names the last page's fingerprint")
for i, lab in enumerate(labels):
    left = 0.4 + i * 2.9
    ax.add_patch(plt.Rectangle((left, 0.45), 2.3, 1.15, fill=False, linewidth=1.6))
    ax.text(left + 1.15, 1.05, lab, ha="center", va="center", fontsize=10)
    if i:
        ax.annotate(
            "",
            xy=(left, 1.0),
            xytext=(left - 0.55, 1.0),
            arrowprops=dict(arrowstyle="->", lw=1.5),
        )
        ax.text(left - 0.28, 1.22, "prev_hash", ha="center", fontsize=8, color="#666")
plt.tight_layout()
plt.show()



In [ ]:
# YOUR TURN 2 -- verifier, then vandalism.
# A chain is clean if:
# 1) each hash starts with the zeros (puzzle was done)
# 2) block i points at the REAL hash of block i-1

def verify_chain(chain, difficulty: int) -> bool:
    prefix = "0" * difficulty
    for i, block in enumerate(chain):
        if not block.hash().startswith(prefix):
            print("block", i, "fails PoW")
            return False
        if i == 0:
            continue
        if block.prev_hash != ???:  # hint: chain[i-1].hash()
            print("block", i, "wrong parent")
            return False
    print("chain looks clean")
    return True

print("before vandalism:", verify_chain(chain, difficulty))

# Quietly change Ava's payment. Do NOT re-mine.
chain[1].txs = "???"   # make it 200 instead of 2
print("after a quiet edit:", verify_chain(chain, difficulty))



## Unbundle (MIT habit)

Bitcoin glued together: a ledger + PoW + P2P gossip + a native coin.

Your class ERC-20 only needs: **balances + signatures + a shared testnet**.
You do not need to mine pizza.

**A2 tonight:** who may mint, what is the cap, who may pause, lost-key plan.

Lab L1 (Naranjo PoW) uses the same verbs.



## Check you understand (say it out loud)

1. What two checks does `verify_chain` run?
2. If you change a payment and do not re-mine, which alarm goes off first?
3. Name one Bitcoin piece Eagecoin does *not* need.

If an answer is a slogan (“decentralized,” “web3,” “the future of money”),
you do not understand it yet. Open the **concept handbook**
(`student-files/site/handbook.html` — or `handbook.html` on the course site)
and rewrite the answer with a cafeteria, key, or lemonade picture.

